# Lab: Batch effects & covariate adjustments

**Course:** Modeling of Complex Biological Systems  
**Tools:** [InMoose](https://github.com/epigenelabs/inmoose) (Integrated Multi Omic Open Source Environment) – Python implementations of ComBat-seq, ComBat-norm, edgeR, DESeq2 

***

## What are batch effects?

Batch effects are systematic technical differences between groups of samples processed at different times, by different technicians, in different environmental conditions, with different protocols, or due to variations in reagents. They are **not** related to the biology of interest, can confound your analysis, and can be mistaken for biological signals if not addressed.

What can be done with batch effects?

- Correction methods: (transformacja danych która redukuje efekty techniczne)

  * linear model adjustments (linear regression)
  * empirical Bayes methods (especially usefull for small datasets)

- Statistical modeling (uwzględnienie efektów technicznych w modelu)
  * batch as a covariate in methods like DESeq2, edgeR
  * surrogate variale analysis (SVA) (technical variables are not known) [see Leek et al., Bioinformatics 28(6), pp. 882-883]

> w obu przypadkach zakładamy 

***

## 1. Data loading and exploration

For this exercise, we will use **pasilla** dataset available in the `InMoose` package, where Brooks et al. (Genome Research, 2011) stidied the effect of RNAi knockdown of Pasilla, the Drosiphila ortolog of mammalian NOVA1 and NOVA2, on the transcriptome. In this dataset, the batch variable is **sequencing type**: single-read vs. paired-end.

In [ ]:
# Install InMoose
!pip install inmoose
!pip install patsy

In [ ]:
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from patsy import dmatrix

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

In [ ]:
from inmoose.data.pasilla import pasilla

# Load pasilla dataset (AnnData: samples x genes)
pas = pasilla()
print(f"Dataset shape: {pas.X.shape[0]} samples x {pas.X.shape[1]} genes")
print(f"\nSample annotations:")
pas.obs

In [ ]:
# Extract count matrix (genes x samples for InMoose functions)
counts = pas.X.T  # genes x samples
anno = pas.obs
gene_names = pas.var.index.tolist()
sample_names = pas.obs.index.tolist()

print(f"Count matrix: {counts.shape[0]} genes x {counts.shape[1]} samples")
print(f"\nCondition: {anno['condition'].value_counts().to_dict()}")
print(f"Batch (type): {anno['type'].value_counts().to_dict()}")

In [ ]:
# Batch x condition cross table
print("Batch x condition:")
print(pd.crosstab(anno['type'], anno['condition']))

In [ ]:
# Filter low-count genes (keep genes with total count > 10)
keep = counts.sum(axis=1) > 10
counts_filt = counts[keep]
gene_names_filt = [g for g, k in zip(gene_names, keep) if k]
print(f"Kept {keep.sum()} of {len(keep)} genes after filtering")

In [ ]:
# Compute log2-CPM for visualization
def log2_cpm(counts):
    """Compute log2 counts per million."""
    lib_sizes = counts.sum(axis=0)
    cpm = counts / lib_sizes * 1e6
    return np.log2(cpm + 1)

log_cpm = log2_cpm(counts_filt)

---

## 2. Visualizing batch effects with PCA

We use Principal Component Analysis (PCA) on the original (uncorrected) data to "roughly" visualize batch effects. We color points by both **condition** (biology) and **batch/type** (technical). When samples group by batch rather than condition (biology), this indicate the presence of batch effects that have to be corrected.

In [ ]:
def pca_plot(data, anno, title='PCA'):
    """PCA plot colored by condition (shape) and batch/type (color)."""
    pca = PCA(n_components=2)
    pcs = pca.fit_transform(data.T)  # samples in rows
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Color by condition
    for cond, color in zip(['treated', 'untreated'], ['red', 'blue']):
        mask = anno['condition'].values == cond
        axes[0].scatter(pcs[mask, 0], pcs[mask, 1], c=color, s=150, 
                       label=cond, edgecolors='black', linewidth=1, zorder=5)
    for i, name in enumerate(anno.index):
        axes[0].annotate(name, (pcs[i, 0], pcs[i, 1]), fontsize=8, ha='center', va='bottom',
                        xytext=(0, 8), textcoords='offset points')
    axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    axes[0].set_title(f'{title} — colored by condition')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Color by batch (sequencing type)
    for batch, color in zip(['single-read', 'paired-end'], ['orange', 'green']):
        mask = anno['type'].values == batch
        axes[1].scatter(pcs[mask, 0], pcs[mask, 1], c=color, s=150, 
                       label=batch, edgecolors='black', linewidth=1, zorder=5)
    for i, name in enumerate(anno.index):
        axes[1].annotate(name, (pcs[i, 0], pcs[i, 1]), fontsize=8, ha='center', va='bottom',
                        xytext=(0, 8), textcoords='offset points')
    axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    axes[1].set_title(f'{title} — colored by batch (seq. type)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return pca

In [ ]:
pca_orig = pca_plot(log_cpm, anno, title='Original data (log2-CPM)')

**Interpretation:**
- If PC1 or PC2 separates samples primarily by **batch** (sequencing type) rather than by **condition** (treated/untreated), then batch effects are dominating the signal.
- Ideally, we want PCA to separate by the biology of interest (condition), not by technical factors.

---

## 3. Methods for correcting batch effects

### Method A: Include batch as a covariate in the linear model

- Instead of correcting the data, include batch as a **covariate** in the statistical model
- The model estimates and removes the batch effect during fitting
- Design formula: `~ batch + condition` (batch effect is modeled but not of interest)

We demonstrate this approach by fitting a linear model to the log-CPM data and visualizing the **residuals** after removing the batch effect.

In [ ]:
#TODO - my tutaj uwzględniamy tylko liniowe zależności !!!

# Regress out batch effect using a linear model
from numpy.linalg import lstsq

# Design matrix with batch only (to remove batch, keep condition signal)
design_batch_only = dmatrix("~ type", data=anno)

# Fit and compute residuals (residuals retain the biological signal)
beta, _, _, _ = lstsq(design_batch_only, log_cpm.T)
batch_effect = (design_batch_only @ beta).T

# Corrected = original - batch effect + grand mean
grand_mean = log_cpm.mean(axis=1, keepdims=True)
log_cpm_lm_corrected = log_cpm - batch_effect + grand_mean

pca_lm_corrected = pca_plot(log_cpm_lm_corrected, anno, title='After linear model batch removal')

### Method B: ComBat

**ComBat** is an empirical Bayes method for batch effect correction developed by Johnson, Li & Rabinovic (Biostatistics, 2007). It models gene expression as a combination of biological signal and batch-specific effects, then uses empirical Bayes shrinkage to borrow information across genes for more robust estimation of these batch parameters (important for small batch sizes). The original ComBat assumes normally distributed data (suitable for microarray or log-transformed values), while ComBat-seq (Zhang et al., NAR Genomics and Bioinformatics 2020) extends the framework to raw RNA-Seq counts using a negative binomial model. Here, we use Python implementation: `pycombat_seq`. It returns adjusted **integer counts** (can be used downstream analysis).

In [ ]:
from inmoose.pycombat import pycombat_seq

batch = list(anno['type'].values) #paired-end, single-read
covar = list(anno['condition'].values) #model covariate

# ComBat-seq: input is genes x samples raw counts
counts_combatseq = pycombat_seq(counts_filt, batch, covar_mod=covar)
# surowe county, jakie zmienne moga na batch wpływać 

print(f"Original counts (gene 0): {counts_filt[0]}")
print(f"Corrected counts (gene 0): {counts_combatseq[0]}")
print(f"\nOutput type: {type(counts_combatseq)}, dtype: {counts_combatseq.dtype}")

In [ ]:
log_cpm_combatseq = log2_cpm(counts_combatseq)
pca_combatseq = pca_plot(log_cpm_combatseq, anno, title='After ComBat-seq')

---

## 4. Integrating batch adjustment in differential expression analysis in DESeq2

The recommended approach for DE analysis is to **include batch as a covariate** in the model design, rather than correcting the data beforehand.

Design: `~ type + condition`
- `type` captures the batch effect (sequencing type)
- `condition` captures the biology of interest (treated vs. untreated)

In [ ]:
# TODO - tutaj od razu jest uwzględniany ten batch

from inmoose.deseq2 import DESeqDataSet, DESeq

# DESeq2 without batch correction
dds_no_batch = DESeqDataSet(countData=pas.X, clinicalData=pas.obs, design="~condition")
dds_no_batch = DESeq(dds_no_batch)
res_no_batch = dds_no_batch.results()
res_no_batch.index = gene_names

# DESeq2 with batch in design
dds_batch = DESeqDataSet(countData=pas.X, clinicalData=pas.obs, design="~type + condition")
dds_batch = DESeq(dds_batch)
res_batch = dds_batch.results()
res_batch.index = gene_names

# Compare
sig_no = set(res_no_batch[res_no_batch['adj_pvalue'] < 0.05].dropna().index)
sig_with = set(res_batch[res_batch['adj_pvalue'] < 0.05].dropna().index)
overlap = sig_no & sig_with
only_no_batch = sig_no - sig_with
only_batch = sig_with - sig_no

print(f"\nDESeq2 without batch adjustment: {len(sig_no)} significant genes (FDR < 0.05)")
print(f"DESeq2 with batch in design:     {len(sig_with)} significant genes (FDR < 0.05)")
print(f"Significant in both:        {len(overlap)}")
print(f"Only without batch adj:     {len(only_no_batch)}")
print(f"Only with batch adj:        {len(only_batch)}")

In [ ]:
# Top results
print(f"\nTop results (without batch adjustment):")
res_no_batch_sorted = res_no_batch.dropna().sort_values('pvalue')
res_no_batch_sorted.head(10)

In [ ]:
print(f"\nTop results (with batch adjustment):")
res_batch_sorted = res_batch.dropna().sort_values('pvalue')
res_batch_sorted.head(10)

### Volcano plot: DESeq2 with batch adjustment

In [ ]:
# Volcano plot for DESeq2 with batch in design
res_plot = res_batch.dropna().copy()
res_plot['-log10_padj'] = -np.log10(res_plot['adj_pvalue'] + 1e-300)

fig, ax = plt.subplots(figsize=(10, 7))

# Classify genes
sig = (res_plot['adj_pvalue'] < 0.05) & (np.abs(res_plot['log2FoldChange']) > 1)
sig_small = (res_plot['adj_pvalue'] < 0.05) & (np.abs(res_plot['log2FoldChange']) <= 1)

ax.scatter(res_plot.loc[~sig & ~sig_small, 'log2FoldChange'], 
          res_plot.loc[~sig & ~sig_small, '-log10_padj'], s=5, alpha=0.3, c='gray')
ax.scatter(res_plot.loc[sig_small, 'log2FoldChange'], 
          res_plot.loc[sig_small, '-log10_padj'], s=5, alpha=0.5, c='orange', label='FDR < 0.05')
ax.scatter(res_plot.loc[sig, 'log2FoldChange'], 
          res_plot.loc[sig, '-log10_padj'], s=8, alpha=0.6, c='red', label='FDR < 0.05, |log2FC| > 1')

ax.axhline(-np.log10(0.05), color='blue', linestyle='--', alpha=0.5)
ax.axvline(1, color='blue', linestyle='--', alpha=0.5)
ax.axvline(-1, color='blue', linestyle='--', alpha=0.5)

ax.set_xlabel('$\\log_2$ Fold Change', fontsize=14)
ax.set_ylabel('$-\\log_{10}$(adjusted P-value)', fontsize=14)
ax.set_title('Volcano Plot — DESeq2 with batch adjustment', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Genes with FDR < 0.05 and |log2FC| > 1: {sig.sum()}")

---

### References

- Zhang, Parmigiani & Johnson (2020). ComBat-Seq: batch effect adjustment for RNA-Seq count data. *NAR Genomics and Bioinformatics* 2(3).
- Johnson, Li & Rabinovic (2007). Adjusting batch effects in microarray expression data. *Biostatistics* 8(1), 118–127.

## Notes 

### Do rozkładów statystycznych p wartości
Rodzaje histogramów dla $p$ wartości 
- jednostajny - wtedy $H_0$ jest prawdziwa
- trzeba też sie zastanowić czy moc testu nei jest za słaba (nie wykrywa nic )

- gdy przypomina rozkład geometryczny - większość 
    - #TODO - tutaj sprawdzić jaka jest probailistyczna interpretacja 

- gdy przypomina bimodalny 
    - mogliśmy źle dobrać test (efekt jest dwustronny) 
    - mamy outliery - małą liczba danych i statystyka testowa zwarca wynik bliżej $H_0$ 

- gdy przymoina geometrycny odwrócony 
    - mogliśmy postawic hipoteze w złą stronę
    - źle dobrany test
    - zrobiliśmy korekcje i wszystkie wartości są przesuniete w pewną stronę 

- rzadkie p value
    - mała próba przy testach nie parametrycznych 
    - zbyt mała liczba boostrapu

- w przypadku wielu testó∑ trzeba zrobić korekcje (nie zależnie od rozkładu) 
    - UWAGA nie zawsze od razu stosujemy korekcje 


bonferonni - zagrawantować oczekiwaną liczbę błędów !!!!
- 


- na później - można dokładnie rozpsiać jak to zachodzi oraz. w jaki sposób to rozumieć 